In [ ]:
# Text Anonymization Benchmark (TAB)

Run the TAB ECHR test documents through all three supported models of a locally running DUUI Anonymize service and write the character offsets expected by TAB's `evaluation.py`.

Start the service before running this notebook (for example, `uvicorn duui_anonymize:app --host 0.0.0.0 --port 9714`). Each model gets a separate output file with the form `{"001-61807": [[54, 62], ...]}`.

In [ ]:
import json
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

DATA_PATH = Path("benchmark/text-anonymization-benchmark/echr_test.json")
OUTPUT_DIR = Path("benchmark/results")
API_URL = "http://anduin.hucompute.org:9714/v1/process"
MODEL_CONFIGS = [
    ("openai/privacy-filter", "openai_privacy-filter", {}),
    ("OpenMed/privacy-filter-nemotron", "openmed_privacy-filter-nemotron", {"trust_remote_code": True}),
    ("bardsai/eu-pii-anonimization-multilang", "bardsai_eu-pii-anonymization-multilang", {}),
]
COMMON_OPTIONS = {"mode": "placeholder"}
TIMEOUT_SECONDS = 300
MAX_RETRIES = 2
LIMIT = None  # Set to a small integer for a smoke test.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"TAB test corpus not found: {DATA_PATH.resolve()}")

with DATA_PATH.open(encoding="utf-8") as stream:
    corpus = json.load(stream)

if not isinstance(corpus, list):
    raise TypeError("The TAB corpus must be a JSON list.")

documents = []
seen_ids = set()
for index, document in enumerate(corpus):
    if not isinstance(document, dict) or not isinstance(document.get("doc_id"), str) or not isinstance(document.get("text"), str):
        raise ValueError(f"Invalid TAB document at index {index}")
    if document["doc_id"] in seen_ids:
        raise ValueError(f"Duplicate doc_id: {document['doc_id']}")
    seen_ids.add(document["doc_id"])
    documents.append((document["doc_id"], document["text"]))

if LIMIT is not None:
    documents = documents[:LIMIT]

print(f"Loaded {len(documents)} TAB test documents ({sum(len(text) for _, text in documents):,} characters).")

In [ ]:
def post_json(url, payload, timeout=TIMEOUT_SECONDS):
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = Request(url, data=body, headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.load(response)
    except HTTPError as error:
        details = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"DUUI returned HTTP {error.code}: {details}") from error
    except URLError as error:
        raise RuntimeError(f"Cannot reach DUUI at {url}: {error.reason}") from error

def process_document(doc_id, text, options):
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = post_json(API_URL, {"text": text, "options": options})
            spans = response.get("detected_spans")
            if not isinstance(spans, list):
                raise ValueError("Response is missing a detected_spans list")
            normalized = []
            for span in spans:
                start, end = span.get("start"), span.get("end")
                if not isinstance(start, int) or not isinstance(end, int) or not 0 <= start < end <= len(text):
                    raise ValueError(f"Invalid span offsets in {doc_id}: {span}")
                item = dict(span)
                item["text"] = text[start:end]  # offsets are authoritative
                normalized.append(item)
            return sorted(normalized, key=lambda span: (span["start"], span["end"]))
        except (RuntimeError, ValueError):
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)


In [ ]:
# Rich API responses are retained for analysis; TAB files contain offsets only.
detected_spans_by_model = {}
tab_predictions_by_model = {}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for model_number, (model, output_name, extra_options) in enumerate(MODEL_CONFIGS, start=1):
    options = {**COMMON_OPTIONS, **extra_options, "model": model}
    output_path = OUTPUT_DIR / f"{output_name}_echr_test.json"
    detected_spans_by_doc = {}
    tab_predictions = {}
    started = time.monotonic()
    print(f"\nModel {model_number}/{len(MODEL_CONFIGS)}: {model}")

    for number, (doc_id, text) in enumerate(documents, start=1):
        spans = process_document(doc_id, text, options)
        detected_spans_by_doc[doc_id] = spans
        tab_predictions[doc_id] = [[span["start"], span["end"]] for span in spans]

        # Save after every document so an interrupted run retains completed work.
        with output_path.open("w", encoding="utf-8") as stream:
            json.dump(tab_predictions, stream, ensure_ascii=False, indent=2)

        elapsed = time.monotonic() - started
        print(f"[{number}/{len(documents)}] {doc_id}: {len(spans)} spans ({elapsed:.1f}s elapsed)")

    detected_spans_by_model[model] = detected_spans_by_doc
    tab_predictions_by_model[model] = tab_predictions
    print(f"Wrote {model} predictions to {output_path.resolve()}")

In [ ]:
for model, predictions in tab_predictions_by_model.items():
    total_spans = sum(map(len, predictions.values()))
    print(f"{model}: {len(predictions)} documents; {total_spans} detected spans")

# Show a compact sample with labels and source text for each model.
for model, spans_by_doc in detected_spans_by_model.items():
    first_doc_id = next(iter(spans_by_doc), None)
    if first_doc_id is not None:
        print(f"\n{model} / {first_doc_id}", spans_by_doc[first_doc_id][:5])

## TAB evaluation

Evaluate all three prediction files with TAB's metrics. This requires the evaluator dependencies (spaCy, NumPy, pandas, intervaltree, and tqdm) and the `en_core_web_md` spaCy model.

In [ ]:
import sys

EVALUATOR_DIR = Path("benchmark/text-anonymization-benchmark").resolve()
if str(EVALUATOR_DIR) not in sys.path:
    sys.path.insert(0, str(EVALUATOR_DIR))

from evaluation import GoldCorpus, evaluate, get_masked_docs_from_file

gold_corpus = GoldCorpus(str(DATA_PATH))
evaluation_by_model = {}

for model, output_name, _ in MODEL_CONFIGS:
    prediction_path = OUTPUT_DIR / f"{output_name}_echr_test.json"
    if not prediction_path.is_file():
        raise FileNotFoundError(f"Missing predictions for {model}: {prediction_path.resolve()}")

    masked_documents = get_masked_docs_from_file(str(prediction_path))
    metrics = evaluate(gold_corpus, masked_documents)
    evaluation_by_model[model] = metrics
    print(f"\n{model}")
    print(json.dumps(metrics, indent=2, sort_keys=True))